In [1]:
import logging
import time
import numpy as np
import pandas as pd
import requests
import json
import csv
import os as os
from datetime import datetime, date
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type, stop_after_attempt, wait_exponential
from metapub import PubMedFetcher, PubMedArticle
from multiprocessing.pool import ThreadPool
from typing import List, Iterator, Optional
from habanero import Crossref


# Configure logging
logging.basicConfig(
    filename='DOI.log',
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

# Initialize the fetcher for PubMed and Crossref
fetcher = PubMedFetcher()
cr = Crossref(mailto="m.n.khanji@umcg.nl")

# Retry logic for handling communication errors
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # Maximum 10 seconds wait.
    wait=wait_fixed(0.4),  # Wait 400ms between retries
    retry=retry_if_exception_type((Exception,))  # Use a tuple for exceptions
)


@retry_on_communication_error()
def fetch_article(pmid: str) -> PubMedArticle:
    """Fetch a single article from PubMed by PMID with logging."""
    logging.info("Fetching article with pmid=%r", pmid)
    t0 = time.perf_counter()
    article = fetcher.article_by_pmid(pmid)
    dt = time.perf_counter() - t0
    logging.info("Fetched pmid=%s in %.3fms", pmid, dt * 1000)

    if article.pmid != pmid:
        logging.warning("Article with pmid=%r returned pmid=%r", pmid, article.pmid)

    return article

@retry_on_communication_error()
def fetch_articles(pmids: List[str], *, processes: Optional[int] = None) -> Iterator[PubMedArticle]:
    """Fetch multiple articles from PubMed in parallel using a thread pool."""
    with ThreadPool(processes=processes) as pool:
        for article in pool.imap_unordered(fetch_article, pmids):
            if article is not None:
                yield article

def save_articles_to_csv(pmids: List[str], csv_file: str):
    """Fetch articles and save them to a CSV file."""
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        out = csv.writer(f)
        first = True
        for article in fetch_articles(pmids, processes=5):
            row = dict(
                pmid=article.pmid,
                pmc=article.pmc,
                title=article.title,
                journal=article.journal,
                doi=article.doi,
                issn=article.issn
            )
            if first:
                out.writerow(row.keys())
                first = False
            out.writerow(row.values())

@retry_on_communication_error()
def publisher_crossref_doi(dois):
    """Fetch publishers for a list of DOIs using Crossref."""
    publishers = []
    for doi in dois:
        try:
            work = cr.works(ids=doi)
            publisher = work["message"].get("publisher")
            publishers.append(publisher)
        except Exception as e:
            logging.error(f"Error: API request failed for doi {doi}: {e}")
            publishers.append(None)
    return publishers

def identify_missing_values(df, column_name):
    """Identify rows with missing values in a specified column."""
    missing_values = df[column_name].isnull()
    return df[missing_values]

@retry_on_communication_error()
def get_publisher_id_from_issn(issn: str) -> str:
    """Query the CrossRef API for a single ISSN and return the publisher ID."""
    url = f"https://api.crossref.org/works?filter=issn:{issn}&select=publisher&mailto=m.n.khanji@umcg.nl"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        data = json.loads(response.text)
        if "message" in data and "items" in data["message"] and data["message"]["items"]:
            first_item = data["message"]["items"][0]
            if isinstance(first_item, dict):
                return list(first_item.values())[0]
    except requests.RequestException as e:
        logging.error(f"Error: API request failed for issn {issn}: {e}")
    return None


def get_publisher_ids_from_issn(missing_df: pd.DataFrame) -> list:
    """Read ISSNs from the missing_df dataframe, query the CrossRef API, and return a list of publisher IDs."""
    issns = missing_df['issn'].tolist()
    publisher_id_list = []
    for issn in issns:
        publisher_id = get_publisher_id_from_issn(issn)
        publisher_id_list.append(publisher_id)
        time.sleep(0.4)  # Sleep to avoid hitting API rate limits
    return publisher_id_list

def add_publishers_to_csv(input_csv: str, output_csv: str):
    """Add publisher information to CSV using DOI and ISSN."""
    # Load the CSV file
    df = pd.read_csv(input_csv)
    dois = df['doi'].tolist()  # Assuming 'doi' is the column name for DOIs

    # Get publishers using DOIs
    publisher_list = publisher_crossref_doi(dois)
    df.loc[:, 'publisher'] = publisher_list

    # Find missing publishers
    missing_df = identify_missing_values(df, 'publisher')

    # Use ISSNs to find missing publishers
    if not missing_df.empty:
        publisher_ids_from_issn = get_publisher_ids_from_issn(missing_df)
        missing_df.loc[:, 'publisher'] = publisher_ids_from_issn

        # Combine the original data with the new data
        df.update(missing_df)

    # Save the final DataFrame to a CSV file
    df.to_csv(output_csv, index=False)

def read_query_from_file(filename):
    """Read and clean query from a file."""
    try:
        with open(filename, 'r') as file:
            query = file.read()
        return query
    except FileNotFoundError:
        logging.error(f"The file '{filename}' was not found.")
        return ""
    except Exception as e:
        logging.error(f"An error occurred while reading the query file: {e}")
        return ""

@retry_on_communication_error()
def get_list(query):
    """Retrieve all PMIDs for a given query using the PubMedFetcher."""
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetcher.pmids_for_query(query,
                                            retstart=start_index,
                                            retmax=num_of_articles,
                                            pmc_only=False)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break
    return pmids

@retry_on_communication_error()
def fetch_pmids_over_period(query_file, start="2000-01-01", stop=None):
    """Fetch PMIDs over a specified period using a query read from a file."""
    query = read_query_from_file(query_file)
    if not query:
        logging.error("Failed to read query.")
        return np.array([])

    if stop is None:
        stop = datetime.now().strftime("%Y-%m-%d")

    start_date_str = start
    pmid_list = []

    while True:
        if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"):
            month_interval = 6
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
            month_interval = 5
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01"):
            month_interval = 4
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01"):
            month_interval = 3
        elif date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01"):
            month_interval = 2
        else:
            month_interval = 4

        next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
        end_date = (next_start - relativedelta(days=1))
        end_date_str = end_date.strftime('%Y-%m-%d')

        date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
        pmids = get_list(date_str + query)
        pmid_list.extend(pmids)
        start_date_str = next_start.strftime('%Y-%m-%d')
        if next_start >= date.fromisoformat(stop):
            break

    # Remove duplicates by converting to a set, then back to a list
    pmid_clean_list = list(set(pmid_list))
    logging.info(f"Total PMIDs fetched: {len(pmid_clean_list)}")

    return np.array(pmid_clean_list)

def save_pmids(pmid_array, directory="PMID_lists"):
    """Save PMIDs to both a text file and a NumPy binary file."""
    # Ensure the directory exists
    os.makedirs(directory, exist_ok=True)

    # Create a date tag for the filename
    date_tag = datetime.now().isoformat()[:10]

    # File paths
    txt_file_path = os.path.join(directory, f'pmids_{date_tag}.txt')
    npy_file_path = os.path.join(directory, f'pmids_{date_tag}.npy')

    # Save PMIDs to a text file
    np.savetxt(txt_file_path, pmid_array, fmt='%s', delimiter=",")
    logging.info(f"PMIDs saved to text file: {txt_file_path}")

    # Save PMIDs to a NumPy binary file
    np.save(npy_file_path, pmid_array)
    logging.info(f"PMIDs saved to binary file: {npy_file_path}")

def main():
    # Define the query file and dates
    query_file = "query"  # Filename containing the PubMed query
    start_date = "2000-01-01"
    stop_date = "2024-08-01"

    # Step 1: Fetch PMIDs over the specified period using a query
    pmid_array = fetch_pmids_over_period(query_file, start=start_date, stop=stop_date)

    # Save PMIDs if any are fetched
    if len(pmid_array) > 0:
        save_pmids(pmid_array)

        # Step 2: Fetch article data from PubMed and save to CSV
        csv_file = "pmid_pmc_title_journal_doi_issn.csv"
        save_articles_to_csv(pmid_array.tolist(), csv_file)

        # Step 3: Add publishers using DOIs and ISSNs
        output_csv = 'FINAL.csv'
        add_publishers_to_csv(csv_file, output_csv)
        logging.info("Process completed.")
    else:
        logging.error("No PMIDs were fetched. Check query or date range.")

if __name__ == "__main__":
    main()


2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO Total PMIDs fetched: 326
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO PMIDs saved to text file: PMID_lists\pmids_2024-08-12.txt
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO PMIDs saved to binary file: PMID_lists\pmids_2024-08-12.npy
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO Fetching article with pmid='39058994'
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO Fetching article with pmid='38701998'
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO Fetching article with pmid='38686630'
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO Fetching article with pmid='38736172'
2024-08-12 17:57:15 LAPTOP-S8N3C7A8 root[28004] INFO Fetching article with pmid='38640546'
2024-08-12 17:57:16 LAPTOP-S8N3C7A8 root[28004] INFO Fetched pmid=39058994 in 580.364ms
2024-08-12 17:57:16 LAPTOP-S8N3C7A8 root[28004] INFO Fetching article with pmid='38589729'
2024-08-12 17:57:16 LAPTOP-S8N3C7A8 root[28004] INFO Fetched pmi